In [24]:
# import necessary libraries
from dotenv import load_dotenv
from agents import Agent, Runner, trace, OpenAIChatCompletionsModel
from agents.mcp import MCPServerStdio
from openai import AsyncAzureOpenAI
from IPython.display import Image, display, Markdown
import os, sys

from agents import set_tracing_disabled
set_tracing_disabled(True)

load_dotenv(override=True)

True

In [2]:
# Let's see if the API key is working/helping us to call LLM from Azure Foundry
import os
# From OpenAI
AZURE_OPENAI_API_KEY= os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_MODEL_DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT")
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")
AZURE_OPENAI_DEPLOYMENT_GPT_41 = os.getenv("AZURE_OPENAI_DEPLOYMENT_GPT_41")
AZURE_OPENAI_DEPLOYMENT_GPT_54_mini = os.getenv("AZURE_OPENAI_DEPLOYMENT_GPT_54_mini")
AZURE_OPENAI_DEPLOYMENT_GPT_55 = os.getenv("AZURE_OPENAI_DEPLOYMENT_GPT_55")
AZURE_OPENAI_DEPLOYMENT_GPT_4O_mini = os.getenv("AZURE_OPENAI_DEPLOYMENT_GPT_4O_mini")
if AZURE_OPENAI_API_KEY:
    print("AZURE_OPENAI_API_KEY is available")
else:
    print("AZURE_OPENAI_API_KEY is not available")


AZURE_OPENAI_API_KEY is available


In [3]:
## setting up the client for OpenAI using Azure Endpoint (Azure Foundry)
client_openai = AsyncAzureOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION
)

## let's a define a model pointing to Azure deployment
model_openai_GPT_54 = OpenAIChatCompletionsModel(
    openai_client=client_openai,
    model=AZURE_OPENAI_DEPLOYMENT_GPT_54_mini
)

In [4]:
"""
On Windows, a stdio MCP Server started from Jupyter kernel weites to a stderr stream with no real life descriptor,
and crashes with io.UnsupportedOperation: fileno. We send the server's stderr to the null device so it always has somewhere real to write,
which lets every cell below use MCPServerStdio exactly as the OpenAI Agents SDK documents it. Mac and Linux doesn't get affected by this.
"""
import functools
import subprocess
import agents.mcp.server 

agents.mcp.server.stdio_client = functools.partial(agents.mcp.server.stdio_client, errlog=subprocess.DEVNULL)

In [5]:
# let's import necessary modules from backend code created by AI Agents from simulating the trading behaviour
from backend.accounts import Account

In [8]:
# let's create an account
account = Account.get("hardeepsingh")
account.reset()
account

Account(name='hardeepsingh', balance=10000.0, strategy='', holdings={}, transactions=[], portfolio_value_time_series=[])

In [11]:
# let's try to buy some shares
account.buy_shares("AMZM", 3, "This ecommerce business looks promising")


Massive API unavailable (No Massive price available for AMZM); using a simulated price
Massive API unavailable (No Massive price available for AMZM); using a simulated price


'Completed. Latest details:\n{"name": "hardeepsingh", "balance": 8581.6189, "strategy": "", "holdings": {"AMZM": 3}, "transactions": [{"symbol": "AMZM", "quantity": 3, "price": 472.7937, "timestamp": "2026-11-03 20:40:53", "rationale": "This ecommerce business looks promising"}], "portfolio_value_time_series": [["2026-11-03 20:38:41", 10000.0], ["2026-11-03 20:41:49", 9997.258899999999]], "total_portfolio_value": 9997.258899999999, "total_profit_loss": -2.741100000001097}'

In [12]:
account.report()

Massive API unavailable (No Massive price available for AMZM); using a simulated price


'{"name": "hardeepsingh", "balance": 8581.6189, "strategy": "", "holdings": {"AMZM": 3}, "transactions": [{"symbol": "AMZM", "quantity": 3, "price": 472.7937, "timestamp": "2026-11-03 20:40:53", "rationale": "This ecommerce business looks promising"}], "portfolio_value_time_series": [["2026-11-03 20:38:41", 10000.0], ["2026-11-03 20:41:49", 9997.258899999999], ["2026-11-03 20:42:52", 9997.408899999999]], "total_portfolio_value": 9997.408899999999, "total_profit_loss": -2.591100000001461}'

In [14]:
account.list_transactions() 

[{'symbol': 'AMZM',
  'quantity': 3,
  'price': 472.7937,
  'timestamp': '2026-11-03 20:40:53',
  'rationale': 'This ecommerce business looks promising'}]

### Now let's create MCP Server and try to call same function using agent

In [19]:
# Now let's use our accounts server as an  MCP server
# sys.executable guarantees the server runs in the notebook's venv (uv run may pick a different env)
params = {
    "command": sys.executable,
    "args": ["-m", "backend.accounts_server"]
}

async with MCPServerStdio(
    params=params,
    client_session_timeout_seconds=30
) as server:
    mcp_tools = await server.list_tools()

In [20]:
mcp_tools

[Tool(name='get_balance', title=None, description='\n    Get the cash balance of the given account name.\n\n    Args:\n        name: The name of the account holder\n\n    Returns:\n        Balance of the account holder as floating number.\n    ', input_schema={'type': 'object', 'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_balanceArguments'}, execution=None, output_schema={'properties': {'result': {'title': 'Result', 'type': 'number'}}, 'required': ['result'], 'title': 'get_balanceOutput', 'type': 'object'}, icons=None, annotations=None, meta=None),
 Tool(name='get_holdings', title=None, description='\n    Get the holdings of the given account name.\n\n    Args:\n        name: The name of the account holder\n\n    Returns:\n        A dictionary containing information on holdings of the holder.\n    ', input_schema={'type': 'object', 'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_holding

In [ ]:
# Use an agent to call these tools hosted in MCP

instructions = "You are able to manage an account for a client, and answer question about the account."
request = "My name is hardeepsingh and my account is under the name hardeepsingh. What's my balance and my holdings?"

In [23]:
async with MCPServerStdio(
    params=params,
    client_session_timeout_seconds=30
) as accounts_mcp_server:
    agent = Agent(
        name = "account_manager",
        instructions=instructions,
        model=model_openai_GPT_54,
        mcp_servers=[accounts_mcp_server],
    )
    result = await Runner.run(
        starting_agent=agent,
        input=request,
    )

    display(Markdown(result.final_output))

Your account details for **hardeepsingh**:

- **Cash balance:** 8581.6189
- **Holdings:** AMZM — 3 shares